<a href="https://colab.research.google.com/github/JonathanDP02/244107020197-Pembelajaran-Mesin/blob/main/JS11_Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# 1. Load Dataset voice.csv
df = pd.read_csv('voice.csv')

# 2. Preprocessing Data
le = LabelEncoder()
df['label'] = le.fit_transform(df['label']) # Mengubah label 'male'/'female' menjadi numerik

X = df.drop('label', axis=1)
y = df['label']

# Melakukan Feature Scaling agar performa SVM optimal
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Konfigurasi Eksperimen
splits = {'70:30 (Train: 70, Test: 30)': 0.3, '80:20 (Train: 80, Test: 20)': 0.2}
kernels = ['linear', 'poly', 'rbf']

results = []

# 4. Iterasi Pelatihan dan Pengujian Model
for split_name, test_size in splits.items():
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=test_size, random_state=42
    )
    for kernel in kernels:
        # Inisialisasi dan latih model SVM
        svm_model = SVC(kernel=kernel, random_state=42)
        svm_model.fit(X_train, y_train)

        # Prediksi dan Evaluasi Akurasi
        y_pred = svm_model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)

        results.append({
            'Rasio Split': split_name,
            'Kernel SVM': kernel.capitalize(),
            'Akurasi': round(acc * 100, 2)
        })

# 5. Menampilkan Hasil dalam Bentuk Tabel
df_results = pd.DataFrame(results)
print("--- TABEL PERFORMANSI KLASIFIKASI SVM ---")
print(df_results.to_string(index=False))

--- TABEL PERFORMANSI KLASIFIKASI SVM ---
                Rasio Split Kernel SVM  Akurasi
70:30 (Train: 70, Test: 30)     Linear    97.06
70:30 (Train: 70, Test: 30)       Poly    95.69
70:30 (Train: 70, Test: 30)        Rbf    98.11
80:20 (Train: 80, Test: 20)     Linear    97.63
80:20 (Train: 80, Test: 20)       Poly    96.85
80:20 (Train: 80, Test: 20)        Rbf    98.26


In [2]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# ==========================================
# 1. FUNGSI EKSTRAKSI FITUR HISTOGRAM & LOAD DATA
# ==========================================
def extract_color_histogram(image_path, bins=(8, 8, 8)):
    # Membaca gambar menggunakan OpenCV
    image = cv2.imread(image_path)
    if image is None:
        return None
    # Menyeragamkan ukuran gambar menjadi 128x128 piksel
    image = cv2.resize(image, (128, 128))
    # Menghitung histogram warna RGB
    hist = cv2.calcHist([image], [0, 1, 2], None, bins, [0, 256, 0, 256, 0, 256])
    # Normalisasi histogram
    cv2.normalize(hist, hist)
    return hist.flatten()

def load_data_from_split(split_dir):
    data = []
    labels = []
    # Folder 'day' dan 'night' di dalam training/test
    classes = ['day', 'night']

    for label_name in classes:
        folder_path = os.path.join(split_dir, label_name)
        if not os.path.exists(folder_path):
            print(f"Peringatan: Folder {folder_path} tidak ditemukan!")
            continue

        for filename in os.listdir(folder_path):
            img_path = os.path.join(folder_path, filename)
            # Memastikan file adalah gambar yang valid
            if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                hist = extract_color_histogram(img_path)
                if hist is not None:
                    data.append(hist)
                    labels.append(label_name)

    return np.array(data), np.array(labels)

# ==========================================
# 2. MEMUAT DATA TRAINING DAN TEST DARI DIREKTORI
# ==========================================
base_path = '/content/images'

train_dir = os.path.join(base_path, 'trainning') # Atau sesuaikan jika namanya 'trainning'
test_dir = os.path.join(base_path, 'test')

# Jika nama foldernya 'trainning' (dengan double n), sesuaikan baris ini:
if not os.path.exists(train_dir) and os.path.exists(os.path.join(base_path, 'trainning')):
    train_dir = os.path.join(base_path, 'trainning')

print("Memuat data training...")
X_train, y_train_raw = load_data_from_split(train_dir)

print("Memuat data test...")
X_test, y_test_raw = load_data_from_split(test_dir)

# Mengubah label teks ('day'/'night') menjadi numerik (0 dan 1)
le = LabelEncoder()
y_train = le.fit_transform(y_train_raw)
y_test = le.transform(y_test_raw)

print(f"Jumlah data training: {len(X_train)}")
print(f"Jumlah data test: {len(X_test)}")

# ==========================================
# 3. FEATURE SCALING
# ==========================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 4. SVM KERNEL RBF & HYPERPARAMETER TUNING
# ==========================================
# Menentukan kombinasi parameter C dan gamma untuk tuning RBF
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.1, 1]
}

# Inisialisasi model dasar SVM dengan kernel RBF[cite: 4]
svc_rbf = SVC(kernel='rbf', random_state=42)

# Menggunakan GridSearchCV dengan Cross-Validation (CV = 5)
grid_search = GridSearchCV(
    estimator=svc_rbf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

print("\nSedang melakukan proses training dan hyperparameter tuning...")
grid_search.fit(X_train_scaled, y_train)

print("\n--- HASIL TUNING HYPERPARAMETER ---")
print("Parameter Terbaik :", grid_search.best_params_)
print(f"Akurasi Terbaik (CV): {grid_search.best_score_ * 100:.2f}%")

# ==========================================
# 5. EVALUASI AKHIR PADA DATA TEST (RASIO 80:20 / Sesuai Folder)
# ==========================================
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)
final_accuracy = accuracy_score(y_test, y_pred)

print("\n--- EVALUASI AKHIR PADA DATA TEST ---")
print(f"Akurasi Pengujian Akhir: {final_accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Memuat data training...
Memuat data test...
Jumlah data training: 240
Jumlah data test: 160

Sedang melakukan proses training dan hyperparameter tuning...

--- HASIL TUNING HYPERPARAMETER ---
Parameter Terbaik : {'C': 1, 'gamma': 'auto'}
Akurasi Terbaik (CV): 98.33%

--- EVALUASI AKHIR PADA DATA TEST ---
Akurasi Pengujian Akhir: 87.50%

Classification Report:
              precision    recall  f1-score   support

         day       0.95      0.79      0.86        80
       night       0.82      0.96      0.89        80

    accuracy                           0.88       160
   macro avg       0.89      0.88      0.87       160
weighted avg       0.89      0.88      0.87       160

